In [1]:
import sys
REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "code"))
from make_network import * 
import numpy as np 
import pandas as pd 
import os 
import pickle


In [2]:
pwd

'/volume_4/research/seongbong/flywire/geosmin_project_version_update/figure8/susceptibility_difference_real_final/3. Redundancy/labial'

In [3]:
rank_df = pd.read_parquet(REPO_ROOT/'figure5/figure5Q,R/data/labial_sweet_rank.parquet')
num_per_concent = dict(zip(rank_df.index.astype(int),np.sum(rank_df<=20,axis=1)))
in_network = rank_df[np.sum(rank_df<=20,axis=1)>=3].index.values.astype(int)

c_to_add = target_ids_valid_all[np.isin(g_info_all,in_network)]
cells_in_network = np.concatenate([c_to_add,labial_sweet,mn9])


In [4]:
metaG_atGRN,originalE_atGRN,metaE_atGRN = make_meta_network(cells_in_network,add=True)
original_G,_,_ = make_network(cells_in_network)


In [5]:
print('# of original edge',len(originalE_atGRN))
print('# of type edge',len(metaG_atGRN.edges()))

# of original edge 1415
# of type edge 255


In [10]:

def num_remaining_edges(originalG):
    num_edges = {}
    for ratio in np.round(np.arange(0,1,0.05),3):        
        num_edges_total = len(original_G.edges())

        this_G_edges = np.array(list(original_G.edges()))
        weights = [original_G[u][v]['weight'] for u,v in original_G.edges()]
        originalE_atGRN = np.array(list(original_G.edges()))

        this_G_edges_sorted = this_G_edges[np.argsort(weights)]
        weights_sorted = np.sort(weights)

        num_edges_to_delete = int(num_edges_total*ratio)
        weights_of_rm_edges = weights_sorted[:num_edges_to_delete]
        rm_edges = [list(x) for x in this_G_edges_sorted[:num_edges_to_delete]]
        num_edges[ratio] =  len(originalE_atGRN) - len(rm_edges)
    return num_edges
remaining_edges = num_remaining_edges(original_G)
pickle.dump(remaining_edges,open('simulation/remaining_edges.pkl','wb'))

In [8]:
from tqdm import tqdm

def remove_edges_weight_order(args):
    originalG,save_path,save_path2 = args  
    for ratio in tqdm(np.round(np.arange(0,1,0.05),3)):        
            num_edges_total = len(original_G.edges())

            this_G_edges = np.array(list(original_G.edges()))
            weights = [original_G[u][v]['weight'] for u,v in original_G.edges()]
            originalE_atGRN = np.array(list(original_G.edges()))

            this_G_edges_sorted = this_G_edges[np.argsort(weights)]
            weights_sorted = np.sort(weights)

            num_edges_to_delete = int(num_edges_total*ratio)
            weights_of_rm_edges = weights_sorted[:num_edges_to_delete]
            rm_edges = [list(x) for x in this_G_edges_sorted[:num_edges_to_delete]]
            #     print(np.sum(determine_recurrent(rm_edges))/len(rm_edges))

            # delete edges from the connectome 
            idx = [x in rm_edges for x in [list(y) for y in originalE_atGRN]]
            edges_to_rm_labial = originalE_atGRN[idx]


            # read original data
            syn_df.reset_index(drop=True,inplace=True)
            syn_df_labial = syn_df.copy()
            with open(f'{save_path2}/synapse_edges_rm_labial_{np.round(ratio*100,2)}%.pkl','wb') as f:
                pickle.dump(rm_edges,f)

            rm_idx = []
            for e in edges_to_rm_labial:
                rm_idx.append(syn_df[(syn_df_labial.Presynaptic_ID==e[0])&(syn_df_labial.Postsynaptic_ID==e[1])].index[0])
            syn_df_labial.drop(rm_idx,inplace=True)
            syn_df_labial.to_parquet(f'{save_path}/synapse_edges_rm_labial_{np.round(ratio*100,2)}%.parquet')

In [12]:
arg_list = []
save_path = 'simulation/data'
save_path2 = 'simulation/rm_edges'

arg_list.append((original_G,save_path,save_path2))
    
    
    

In [13]:
import os
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
import matplotlib.pyplot as plt


n_proc = 20 
with Pool(processes=n_proc) as pool:
    # imap에 tqdm을 감싸서 진행상황 표시
    results = list(tqdm(
        pool.imap(remove_edges_weight_order, arg_list),
        total=len(arg_list)
    ))


100%|████████████████████████████████████████████| 1/1 [09:32<00:00, 572.84s/it]
